# Module 3: Data Preprocessing and Data Quality
**Purpose:** Check missing values and duplicates, handle missing values, and inspect outliers. The original notebook reports duplicate counts; it does not actually drop duplicate rows.

## Shared setup — load the original project datasets
This setup is included so this notebook can run independently in a fresh Google Colab session.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

base_url = 'https://raw.githubusercontent.com/Dinesh2006-Dudi/ML-Datasets/main/archive/'
matches = pd.read_csv(base_url + 'matches.csv')
deliveries = pd.read_csv(base_url + 'deliveries.csv')
batting_stats = pd.read_csv(base_url + 'batting_stats.csv')
bowling_stats = pd.read_csv(base_url + 'bowling_stats.csv')
venues = pd.read_csv(base_url + 'venues.csv')
squads = pd.read_csv(base_url + 'squads.csv')
print('Matches:', matches.shape, '| Deliveries:', deliveries.shape)

                                      Data Preprocessing

In [ ]:
matches.isnull().sum()

In [ ]:
deliveries.isnull().sum()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

base_url = "https://raw.githubusercontent.com/Dinesh2006-Dudi/ML-Datasets/main/archive/"
if 'matches' not in locals() and 'matches' not in globals():
    matches = pd.read_csv(base_url + "matches.csv")
if 'deliveries' not in locals() and 'deliveries' not in globals():
    deliveries = pd.read_csv(base_url + "deliveries.csv")

# Missing values for matches and deliveries
matches_missing = matches.isnull().sum()
deliveries_missing = deliveries.isnull().sum()

# Duplicate values
matches_duplicates = matches.duplicated().sum()
matches_unique = len(matches) - matches_duplicates

deliveries_duplicates = deliveries.duplicated().sum()
deliveries_unique = len(deliveries) - deliveries_duplicates

# Visualization
plt.figure(figsize=(12, 8))

# Subplot 1: Matches Missing Values
plt.subplot(2, 2, 1)
matches_filtered = matches_missing[matches_missing > 0]
if len(matches_filtered) > 0:
    matches_filtered.plot(kind='bar')
    plt.ylabel("Count")
else:
    plt.text(0.5, 0.5, "No Missing Values", ha='center', va='center', fontsize=12, color='green')
plt.title("Matches: Columns with Missing Values")
plt.xticks(rotation=45, ha='right')

# Subplot 2: Deliveries Missing Values
plt.subplot(2, 2, 2)
deliveries_filtered = deliveries_missing[deliveries_missing > 0]
if len(deliveries_filtered) > 0:
    deliveries_filtered.plot(kind='bar')
    plt.ylabel("Count")
else:
    plt.text(0.5, 0.5, "No Missing Values", ha='center', va='center', fontsize=12, color='green')
plt.title("Deliveries: Columns with Missing Values")
plt.xticks(rotation=45, ha='right')

# Subplot 3: Matches Duplicates
plt.subplot(2, 2, 3)
plt.bar(["Unique", "Duplicate"], [matches_unique, matches_duplicates], color=['blue', 'orange'])
plt.title("Matches: Unique vs Duplicate Rows")
plt.ylabel("Count")

# Subplot 4: Deliveries Duplicates
plt.subplot(2, 2, 4)
plt.bar(["Unique", "Duplicate"], [deliveries_unique, deliveries_duplicates], color=['blue', 'orange'])
plt.title("Deliveries: Unique vs Duplicate Rows")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

Handling of matches dataset

In [ ]:
matches['first_ings_score'] = matches['first_ings_score'].fillna(matches['first_ings_score'].median())
matches['first_ings_wkts'] = matches['first_ings_wkts'].fillna(matches['first_ings_wkts'].median())
matches['second_ings_score'] = matches['second_ings_score'].fillna(matches['second_ings_score'].median())
matches['second_ings_wkts'] = matches['second_ings_wkts'].fillna(matches['second_ings_wkts'].median())
matches['balls_left'] = matches['balls_left'].fillna(0)

matches['wb_runs'] = matches['wb_runs'].fillna(0)
matches['wb_wickets'] = matches['wb_wickets'].fillna(0)

matches['player_of_the_match'] = matches['player_of_the_match'].fillna('Unknown')
matches['top_scorer'] = matches['top_scorer'].fillna('Unknown')
matches['best_bowling'] = matches['best_bowling'].fillna('Unknown')

matches.dropna(subset=['match_winner'], inplace=True)
matches['wb_runs'].fillna(0, inplace=True)

In [ ]:
matches.isnull().sum()

Handling of deliveries dataset

In [ ]:
deliveries['wicket_type'] = deliveries['wicket_type'].fillna('No Wicket')
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('Not Out')
deliveries['fielder'] = deliveries['fielder'].fillna('No Fielder')

deliveries['wide'] = deliveries['wide'].fillna(0)
deliveries['noballs'] = deliveries['noballs'].fillna(0)
deliveries['legbyes'] = deliveries['legbyes'].fillna(0)

In [ ]:
deliveries.isnull().sum()

Duplicates removal

In [ ]:
matches.duplicated().sum()

In [ ]:
deliveries.duplicated().sum()

Outlier Detection

In [ ]:
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    outliers = df[(df[column] < lower) | (df[column] > upper)]

    print(f"--- {column} ---")
    print(f"Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
    print(f"Normal Range: {lower:.1f} to {upper:.1f}")
    print(f"Outliers: {len(outliers)} found\n")

    # --- Simple Box Plot Add chesa ---
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df[column].dropna(), color='lightblue')
    plt.title(f'Boxplot - {column} (Outliers = {len(outliers)})')
    plt.xlabel(column)
    plt.show()

    return outliers


detect_outliers(matches, 'first_ings_score')
detect_outliers(matches, 'wb_runs')
detect_outliers(matches, 'wb_wickets')
detect_outliers(matches, 'balls_left')